# Stage 7 - Agent + MLflow monitoring

Two things in this notebook:
1. A **query router / agent** built with DSPy (same as Lab 7) that reads the question and picks the right tool
2. **MLflow** to log every query so we have a record of latency, tool used, answer length etc.

The three tools the agent can call:
- `qa` → RAG: retrieve matching comments, answer with ollama
- `summarize` → same retrieval but with a summary prompt
- `sentiment` → just counts from the dataframe, no LLM needed

DSPy handles the classification. We use `dspy.Predict` with a `Signature` - same pattern as the last part of Lab 7.

In [1]:
import sys
sys.path.append("../src")

import dspy
import agent
import monitoring

c:\Users\Saeed\Documents\370\notebooks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/06/07 02:32:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/07 02:32:38 INFO mlflow.store.db.utils: Updating database tables
2026/06/07 02:32:40 INFO mlflow.tracking.fluent: Experiment with name 'youtube-rag' does not exist. Creating a new experiment.


## The DSPy classifier

The router in `agent.py` is a DSPy `Predict` module with a custom `Signature`.
DSPy formats the prompt and parses the output automatically - we just define what goes in and what we want out.

In [2]:
# quick look at what the DSPy signature looks like
print(agent.IntentSignature.__doc__)

Classify a user question about YouTube comments.
    Output exactly one of: qa, summarize, sentiment
    - qa        : asking a specific question about what commenters said
    - summarize : wants a general overview or summary of opinions
    - sentiment : wants to know how positive/negative the comments are


In [3]:
# test the router on a few examples - should pick the right tool
test_questions = [
    "summarize what people say about the screen",
    "how positive are the comments overall?",
    "did anyone mention overheating?",
    "what do people think about the price?",
]

for q in test_questions:
    intent = agent.route(q)
    print(f"{intent:12s} <- {q}")

summarize    <- summarize what people say about the screen
sentiment    <- how positive are the comments overall?
sentiment    <- did anyone mention overheating?
sentiment    <- what do people think about the price?


## Full agent - route + answer

In [4]:
out = agent.handle("summarize the main complaints people have")
print("intent:", out["intent"])
print()
print(out["answer"])

reusing existing index (18771 comments)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5722.26it/s]


intent: summarize

Here is a 4-5 sentence summary covering the main points, overall mood, and anything that keeps coming up:

The comments on this YouTube video express frustration and anger towards a new product or service, with many users feeling let down by its performance. The issues mentioned include poor battery life, slow performance, and lack of features. A common thread throughout the comments is the perception that consumers are being taken advantage of, with some users expressing outrage at the idea of paying for a product that doesn't meet their expectations. Many commenters also mention that they have had positive experiences with similar products in the past, leading to feelings of disappointment and frustration. The overall mood is one of anger and disillusionment, with many users feeling like they've been misled or taken advantage of.


In [5]:
out = agent.handle("how positive are people overall?")
print(out["answer"])

sentiment breakdown over 18,771 comments:
  neutral: 10935 (58%)
  positive: 5764 (31%)
  negative: 2072 (11%)


In [6]:
out = agent.handle("did anyone mention the price being too high?")
print("intent:", out["intent"])
print(out["answer"])
if out["sources"]:
    print("\ncomments it used:")
    for doc, meta in out["sources"][:3]:
        print(f"  [{meta.get('sentiment', '?')}] {doc[:100]}")

intent: qa
Yes, several commenters mentioned that the price is too high. 

- higher price
- no joke but it is basically a overpriced 
- why price is increased to ask now ? for phone 16

comments it used:
  [neutral] higher price
  [neutral] the price literally hasn’t changed
  [neutral] i think the is not that bad. the real problem is overpriced


## MLflow logging

Wrap every `agent.handle()` call with `monitoring.logged_query()` and MLflow records the run.
After running these cells, open a terminal and run `mlflow ui` then go to http://localhost:5000.

In [7]:
out = monitoring.logged_query("what do people like about the design?", agent.handle)
print(out["answer"][:300])

Here is a 4-5 sentence summary covering the main points, overall mood, and anything that keeps coming up:

The comments on the YouTube video are overwhelmingly negative towards the new design of a phone. Many users express their dislike for the design, calling it "ugly", "weird", and "boring". Howev


In [8]:
out = monitoring.logged_query("summarize opinions on the camera quality", agent.handle)
print(out["answer"][:300])

Here is a summary of the main points, overall mood, and recurring themes in the YouTube comments:

The comments revolve around the camera quality of a specific phone model. Viewers are asking for reviews, comparisons, and demonstrations of the camera's performance, with many expressing interest in t


## Evaluation run

Run a fixed set of questions and log the averages - latency, number of retrieved sources, answer length.
This is our 'monitoring' piece - run it again later to check if anything drifts.

In [ ]:
eval_questions = [
    "what do people think about the battery?",
    "summarize opinions on the camera",
    "how negative are the comments?",
    "did anyone mention the price being too high?",
    "what are the most common complaints?",
    "do people like the build quality?",
]

monitoring.evaluate(agent.handle, eval_questions)
print("done - open mlflow ui to see the results")

## DSPy inspection - what the model actually sees

One nice thing about DSPy is you can inspect the prompt history to see exactly what was sent to the model.

In [ ]:
# run a classification and inspect what DSPy sent under the hood
_ = agent.route("what do people think about the speakers?")

# show the last prompt + response
import agent as _agent_mod
try:
    history = agent.lm.history
    if history:
        last = history[-1]
        print("prompt sent:")
        print(last.get("prompt", last.get("messages", "")))
        print("\nmodel response:", last.get("response", ""))
except Exception as e:
    print("can't inspect history:", e)

---
MLflow UI: run `mlflow ui` in the terminal, then open http://localhost:5000